# 02b — TBIO-8110 recover existing Proseg outputs without rerunning inference

Use this notebook when Proseg's durable outputs were written successfully but
the parallel notebook failed during the Python post-processing stage.

The current failure report contains:

```text
NameError: name 'process_sample' is not defined
```

for all 12 Ada samples. That error occurs after the expensive Proseg inference
phase and does not indicate failed segmentation or a directory mismatch.

This recovery notebook never launches the Proseg executable. For each sample,
it validates and reuses:

```text
*_proseg_qcclass_counts.mtx.gz
*_proseg_qcclass_cell_metadata.csv.gz
*_proseg_qcclass_gene_metadata.csv.gz
*_proseg_qcclass_cell_polygons.geojson.gz
```

It then generates:

```text
*_proseg_qcclass_cdata.h5ad
*_proseg_qcclass_cell_boundaries.parquet
*_proseg_qcclass_metrics_cells.csv
*_proseg_qcclass_qc_panels.pdf
*_proseg_qcclass_boundaries_whole.png
*_proseg_qcclass_boundaries_preview.png
*_proseg_qcclass_SUCCESS.json
```

Already complete post-processed samples are reused unless
`OVERWRITE_POSTPROCESS=True`.


In [1]:
# ---------------------------------------------------------------------
# Imports and configuration
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import gzip
import hashlib
import json
import os
import sys
import shlex
import shutil
import subprocess
import tempfile
import time
import traceback
import warnings
import zlib
from pathlib import Path

import anndata as ad
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import spatialdata
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from scipy.io import mmread
from shapely.affinity import affine_transform
from shapely.geometry import box

Image.MAX_IMAGE_PIXELS = None

CONFIG_PATH = Path(
    os.environ.get(
        "VISIUMHD_PIPELINE_CONFIG",
        "/stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/"
        "derived_files/tbio8110_stardist_proseg_resolvi_v1/"
        "00_config/pipeline_config.json",
    )
)
CONFIG = json.loads(CONFIG_PATH.read_text())
CUSTOM_FUNCTION_DIR = Path(CONFIG["paths"]["functiondirs"])
if CUSTOM_FUNCTION_DIR.exists():
    if str(CUSTOM_FUNCTION_DIR) not in sys.path:
        sys.path.insert(0, str(CUSTOM_FUNCTION_DIR))
    print("Custom function directory enabled:", CUSTOM_FUNCTION_DIR)
else:
    warnings.warn(
        f"Configured functiondirs path does not exist: {CUSTOM_FUNCTION_DIR}. "
        "The built-in notebook helpers will be used."
    )
DERIVED_ROOT = Path(CONFIG["paths"]["derived_root"])
TEMP_ROOT = Path(CONFIG["paths"]["temp_root"])
CONFIG_ROOT = Path(CONFIG["paths"]["config_root"])
manifest = pd.read_csv(CONFIG_ROOT / "sample_manifest.csv")

PRIOR_TEMP_ROOT = TEMP_ROOT / "01_stardist_qcprior"
PRIOR_DERIVED_ROOT = DERIVED_ROOT / "01_stardist_qcprior"
PROSEG_TEMP_ROOT = TEMP_ROOT / "02_proseg"
PROSEG_DERIVED_ROOT = DERIVED_ROOT / "02_proseg"
PROSEG_TEMP_ROOT.mkdir(parents=True, exist_ok=True)
PROSEG_DERIVED_ROOT.mkdir(parents=True, exist_ok=True)

PROSEG_EXE = Path(
    "/home/domino/reny28/Projects/CosMX_projects/"
    "P06364_CRC_DS20250227_48371/code/Bash/.tools/bin/proseg"
)

# Proseg is CPU-multithreaded. Four concurrent jobs should divide host CPU
# capacity rather than each requesting 32 threads.
MAX_PARALLEL_PROSEG = int(
    os.environ.get("PROSEG_MAX_PARALLEL", "4")
)
if MAX_PARALLEL_PROSEG < 1:
    raise ValueError("MAX_PARALLEL_PROSEG must be >= 1.")

AVAILABLE_CPU_THREADS = os.cpu_count() or 1
DEFAULT_THREADS_PER_PROSEG = max(
    1,
    min(
        16,
        AVAILABLE_CPU_THREADS // MAX_PARALLEL_PROSEG,
    ),
)
NTHREADS = int(
    os.environ.get(
        "PROSEG_NTHREADS",
        str(DEFAULT_THREADS_PER_PROSEG),
    )
)
if NTHREADS < 1:
    raise ValueError("NTHREADS must be >= 1.")

PARALLEL_POLL_SECONDS = 15
PROSEG_EXTRA_ARGS = []
RUN_PROSEG = False
OVERWRITE_QC_INPUT = False
OVERWRITE_PROSEG = False
OVERWRITE_POSTPROCESS = False
CONTINUE_ON_ERROR = True

INCLUDE_TRANSCRIPT_POINTS_IN_ZARR = False
WRITE_EXPECTED_COUNTS = False
H5AD_COMPRESSION = "lzf"
MAKE_QC_PDF = True
MAKE_OVERLAYS = True
OVERLAY_ERRORS_ARE_FATAL = False
WHOLE_IMAGE_MAX_SIDE = 5000
PREVIEW_CROP_SIZE_PX = 3500
PIPELINE_VERSION = "reusable-proseg-durable-output-v1"

print("Recovery-only mode: Proseg inference will NOT be launched.")
print("Available CPU threads:", AVAILABLE_CPU_THREADS)
print("Concurrent Proseg processes:", MAX_PARALLEL_PROSEG)
print("Threads per Proseg process:", NTHREADS)
print(
    "Maximum requested Proseg threads:",
    MAX_PARALLEL_PROSEG * NTHREADS,
)
print("Samples:", manifest["sample"].astype(str).tolist())


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_

Custom function directory enabled: /host_root/nethome/reny28/Projects/Custom_functions/python_functions
Recovery-only mode: Proseg inference will NOT be launched.
Available CPU threads: 96
Concurrent Proseg processes: 4
Threads per Proseg process: 16
Maximum requested Proseg threads: 64
Samples: ['Ada-1', 'Ada-3R', 'Ada-4R', 'Ada-6', 'Ada-7', 'Ada-8', 'Ada-9', 'Ada-11R', 'Ada-12', 'Ada-14R', 'Ada-15', 'Ada-16']


In [2]:
# ---------------------------------------------------------------------
# Paths and low-level utilities
# ---------------------------------------------------------------------
def paths_for_sample(row: pd.Series) -> dict[str, Path]:
    sample = str(row["sample"])
    prior_temp = PRIOR_TEMP_ROOT / sample
    prior_durable = PRIOR_DERIVED_ROOT / sample
    temp = PROSEG_TEMP_ROOT / sample
    durable = PROSEG_DERIVED_ROOT / sample
    temp.mkdir(parents=True, exist_ok=True)
    durable.mkdir(parents=True, exist_ok=True)

    return {
        "sample": sample,
        "raw_matrix": Path(row["raw_matrix"]),
        "aligned_qc": Path(row["aligned_qc_parquet"]),
        "image_crop": prior_temp / f"{sample}_he_crop_0p3mpp.tiff",
        "prior_mask": prior_temp / f"{sample}_stardist_qcfiltered_uint32.npy",
        "prior_audit": prior_durable / f"{sample}_stardist_qc_label_audit.csv.gz",
        "prior_metadata": prior_durable / f"{sample}_stardist_qc_metadata.json",
        "qc_input": temp / f"{sample}_qcclass_proseg_input.zarr",
        "qc_input_summary": temp / f"{sample}_qcclass_proseg_input_summary.json",
        "native_zarr": temp / f"{sample}_proseg_native.zarr",
        "log": temp / f"{sample}_proseg_qcclass_run.log",
        "command": durable / f"{sample}_proseg_qcclass_command.txt",
        "counts": durable / f"{sample}_proseg_qcclass_counts.mtx.gz",
        "expected_counts": durable / f"{sample}_proseg_qcclass_expected_counts.mtx.gz",
        "cell_metadata": durable / f"{sample}_proseg_qcclass_cell_metadata.csv.gz",
        "gene_metadata": durable / f"{sample}_proseg_qcclass_gene_metadata.csv.gz",
        "polygons": durable / f"{sample}_proseg_qcclass_cell_polygons.geojson.gz",
        "run_status": durable / f"{sample}_proseg_qcclass_run_status.json",
        "cdata": durable / f"{sample}_proseg_qcclass_cdata.h5ad",
        "boundaries": durable / f"{sample}_proseg_qcclass_cell_boundaries.parquet",
        "metrics": durable / f"{sample}_proseg_qcclass_metrics_cells.csv",
        "qc_pdf": durable / f"{sample}_proseg_qcclass_qc_panels.pdf",
        "overlay_whole": durable / f"{sample}_proseg_qcclass_boundaries_whole.png",
        "overlay_preview": durable / f"{sample}_proseg_qcclass_boundaries_preview.png",
        "overlay_error": durable / f"{sample}_proseg_qcclass_overlay_ERROR.json",
        "success": durable / f"{sample}_proseg_qcclass_SUCCESS.json",
        "failure": durable / f"{sample}_proseg_qcclass_FAILURE.json",
    }


def atomic_json(payload, path: Path):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str))
    temp.replace(path)


def sanitize_none(value):
    if value is None:
        return ""
    if isinstance(value, dict):
        return {str(k): sanitize_none(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_none(v) for v in value]
    if isinstance(value, tuple):
        return tuple(sanitize_none(v) for v in value)
    return value


def ensure_integer_csr(X, tolerance=1e-6):
    X = sp.csr_matrix(X)
    X.sum_duplicates()
    X.eliminate_zeros()
    X.sort_indices()
    if X.data.size:
        error = float(np.max(np.abs(X.data - np.rint(X.data))))
        if error > tolerance:
            raise ValueError(
                f"Count matrix is not integer-like; max deviation={error}"
            )
        X.data = np.rint(X.data).astype(np.uint32, copy=False)
    return X


def add_standard_qc(adata: ad.AnnData):
    if "gene" in adata.var.columns:
        names = pd.Index(adata.var["gene"].astype(str).str.upper())
    else:
        names = pd.Index(adata.var_names.astype(str).str.upper())

    adata.var["mt"] = np.asarray(
        names.str.startswith(("MT-", "MT_")), dtype=bool
    )
    adata.var["ribo"] = np.asarray(
        names.str.startswith(("RPS", "RPL")), dtype=bool
    )
    adata.var["hb"] = np.asarray(
        names.str.match(r"^HB(?!P)"), dtype=bool
    )
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo", "hb"],
        percent_top=None,
        log1p=False,
        inplace=True,
    )
    adata.obs["n_counts"] = adata.obs["total_counts"].to_numpy()
    adata.obs["n_genes"] = adata.obs["n_genes_by_counts"].to_numpy()


In [3]:
# ---------------------------------------------------------------------
# Build/reuse QC-filtered integer AnnData input for Proseg
# ---------------------------------------------------------------------
def build_qc_input(row: pd.Series, p: dict[str, Path]) -> Path:
    sample = str(row["sample"])
    aligned = pd.read_parquet(p["aligned_qc"])
    keep_barcodes = pd.Index(
        aligned.loc[aligned["qc_keep"].astype(bool), "barcode"].astype(str)
    )

    expected_summary = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        "raw_matrix": str(p["raw_matrix"]),
        "n_requested_qc_keep_barcodes": int(len(keep_barcodes)),
    }

    if (
        p["qc_input"].exists()
        and p["qc_input_summary"].exists()
        and not OVERWRITE_QC_INPUT
    ):
        existing = json.loads(p["qc_input_summary"].read_text())
        if all(existing.get(k) == v for k, v in expected_summary.items()):
            check = ad.read_zarr(p["qc_input"])
            if check.n_obs == existing["n_bins_written"]:
                print("Reusing QC input:", p["qc_input"])
                return p["qc_input"]
        raise RuntimeError(
            "Existing QC input does not match current configuration. Set "
            "OVERWRITE_QC_INPUT=True to rebuild it."
        )

    if p["qc_input"].exists():
        shutil.rmtree(p["qc_input"])

    print("Reading raw 2-µm matrix:", p["raw_matrix"])
    bins = sc.read_10x_mtx(
        p["raw_matrix"],
        var_names="gene_symbols",
        make_unique=True,
        cache=False,
        gex_only=True,
    )
    bins.obs_names = bins.obs_names.astype(str)
    selected = bins.obs_names.isin(keep_barcodes)
    overlap_fraction = float(selected.sum() / max(len(keep_barcodes), 1))
    if overlap_fraction < 0.95:
        raise RuntimeError(
            f"Only {overlap_fraction:.3%} of requested QC-kept barcodes "
            "were found in the raw matrix."
        )

    subset = bins[selected].copy()
    subset.X = ensure_integer_csr(subset.X)

    aligned_indexed = aligned.set_index("barcode").reindex(subset.obs_names)
    if aligned_indexed[
        ["proseg_x_um", "proseg_y_um"]
    ].isna().any(axis=1).any():
        raise RuntimeError("Selected raw barcodes lack aligned coordinates.")

    obs = pd.DataFrame(index=subset.obs_names.copy())
    for column in ["qc_class_raw", "qc_class", "qc_keep"]:
        obs[column] = aligned_indexed[column].to_numpy()

    qc_input = ad.AnnData(
        X=subset.X,
        obs=obs,
        var=subset.var.copy(),
        obsm={
            "spatial": aligned_indexed[
                ["proseg_x_um", "proseg_y_um"]
            ].to_numpy(dtype=np.float32)
        },
    )
    qc_input.var_names = subset.var_names.astype(str)
    qc_input.var_names_make_unique()
    qc_input.uns["proseg_qc_input"] = sanitize_none(
        {
            **expected_summary,
            "n_bins_written": int(qc_input.n_obs),
            "n_genes": int(qc_input.n_vars),
            "barcode_overlap_fraction": overlap_fraction,
            "coordinate_convention": (
                "obsm['spatial'][:,0]=Space Ranger fullres row*mpp; "
                "[:,1]=fullres column*mpp"
            ),
        }
    )

    qc_input.write_zarr(p["qc_input"])
    summary = {
        **expected_summary,
        "n_bins_written": int(qc_input.n_obs),
        "n_genes": int(qc_input.n_vars),
        "barcode_overlap_fraction": overlap_fraction,
    }
    atomic_json(summary, p["qc_input_summary"])

    del bins, subset, qc_input
    gc.collect()
    return p["qc_input"]


In [4]:
# ---------------------------------------------------------------------
# Proseg command and execution
# ---------------------------------------------------------------------
def clean_transform(values, zero_tolerance=1e-6):
    cleaned = []
    for value in values:
        value = float(value)
        if abs(value) < zero_tolerance:
            value = 0.0
        cleaned.append(value)
    if len(cleaned) != 3:
        raise ValueError(cleaned)
    if any(value < 0 for value in cleaned):
        raise ValueError(
            "A meaningful negative transform coefficient remains. The current "
            "Proseg CLI may reject it; inspect image orientation."
        )
    return [f"{value:.12g}" for value in cleaned]


def build_command(p, qc_input, prior_meta):
    x_cli = clean_transform(prior_meta["x_transform"])
    y_cli = clean_transform(prior_meta["y_transform"])

    cmd = [
        str(PROSEG_EXE),
        "--anndata",
        "--anndata-coordinate-key",
        "spatial",
        "--nthreads",
        str(NTHREADS),
        "--burnin-voxel-size",
        "2",
        "--voxel-size",
        "2",
        "--voxel-layers",
        "1",
        "--ignore-z-coord",
        "--diffusion-sigma-near",
        "2",
        "--diffusion-sigma-far",
        "8",
        "--cellpose-masks",
        str(p["prior_mask"]),
        "--cellpose-x-transform",
        *x_cli,
        "--cellpose-y-transform",
        *y_cli,
        *[str(v) for v in PROSEG_EXTRA_ARGS],
        "--output-spatialdata",
        str(p["native_zarr"]),
        "--output-cell-metadata",
        str(p["cell_metadata"]),
        "--output-gene-metadata",
        str(p["gene_metadata"]),
        "--output-counts",
        str(p["counts"]),
        "--output-cell-polygons",
        str(p["polygons"]),
    ]
    if WRITE_EXPECTED_COUNTS:
        cmd.extend(
            ["--output-expected-counts", str(p["expected_counts"])]
        )
    if not INCLUDE_TRANSCRIPT_POINTS_IN_ZARR:
        cmd.append("--exclude-spatialdata-transcripts")
    if OVERWRITE_PROSEG:
        cmd.append("--overwrite")
    cmd.append(str(qc_input))
    return cmd


def run_with_log(cmd, log_path: Path):
    print("Running:\n", shlex.join(cmd))
    started = time.time()
    with open(log_path, "w", buffering=1) as log_handle:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
        return_code = process.wait()
    print(
        f"Return code={return_code}; elapsed="
        f"{(time.time() - started) / 60:.1f} min"
    )
    return return_code


In [5]:
# ---------------------------------------------------------------------
# Durable-output validation and recovery
# ---------------------------------------------------------------------
def decompress_first_complete_gzip_member(
    source: Path,
    destination: Path,
    chunk_bytes=8 * 1024 * 1024,
):
    decoder = zlib.decompressobj(16 + zlib.MAX_WBITS)
    total_in = total_out = 0
    trailing_prefix = b""
    with source.open("rb") as src, destination.open("wb") as dst:
        while True:
            chunk = src.read(chunk_bytes)
            if not chunk:
                break
            total_in += len(chunk)
            decoded = decoder.decompress(chunk)
            if decoded:
                dst.write(decoded)
                total_out += len(decoded)
            if decoder.eof:
                trailing_prefix = decoder.unused_data[:64]
                break
        if not decoder.eof:
            raise EOFError("First gzip member is incomplete.")
        flushed = decoder.flush()
        if flushed:
            dst.write(flushed)
            total_out += len(flushed)
        trailing_bytes = len(decoder.unused_data) + sum(
            len(x) for x in iter(lambda: src.read(chunk_bytes), b"")
        )
    return {
        "method": "first_complete_gzip_member",
        "compressed_bytes_consumed": total_in,
        "decompressed_bytes": total_out,
        "ignored_trailing_bytes": trailing_bytes,
        "ignored_trailing_prefix_hex": trailing_prefix.hex(),
    }


def read_proseg_counts(path: Path):
    try:
        with gzip.open(path, "rb") as handle:
            X = mmread(handle).tocsr()
        report = {"method": "normal_gzip"}
    except Exception as exc:
        warnings.warn(
            f"Normal Matrix Market read failed; trying first complete gzip "
            f"member: {exc}"
        )
        with tempfile.NamedTemporaryFile(suffix=".mtx", delete=False) as temp:
            temp_path = Path(temp.name)
        try:
            report = decompress_first_complete_gzip_member(path, temp_path)
            X = mmread(temp_path).tocsr()
        finally:
            temp_path.unlink(missing_ok=True)
    return ensure_integer_csr(X), report


def clear_local_crs(gdf: gpd.GeoDataFrame, context: str):
    gdf = gdf.copy()
    if gdf.crs is not None:
        print(
            f"{context}: discarding CRS {gdf.crs}; coordinates are local microns."
        )
        try:
            gdf = gdf.set_crs(None, allow_override=True)
        except Exception:
            gdf.crs = None
    return gdf


def read_gz_geojson(path: Path):
    with tempfile.NamedTemporaryFile(
        suffix=".geojson", delete=False
    ) as temp:
        temp_path = Path(temp.name)
        with gzip.open(path, "rb") as source:
            shutil.copyfileobj(source, temp)
    try:
        gdf = gpd.read_file(temp_path)
    finally:
        temp_path.unlink(missing_ok=True)
    return clear_local_crs(gdf, f"Read {path.name}")


def validate_separate_outputs(p):
    required = [
        p["counts"],
        p["cell_metadata"],
        p["gene_metadata"],
        p["polygons"],
    ]
    missing = [
        str(path)
        for path in required
        if not path.exists() or path.stat().st_size == 0
    ]
    if missing:
        raise FileNotFoundError(
            "Missing/empty durable Proseg outputs:\n" + "\n".join(missing)
        )

    X, read_report = read_proseg_counts(p["counts"])
    cells = pd.read_csv(p["cell_metadata"])
    genes = pd.read_csv(p["gene_metadata"])
    if X.shape == (len(genes), len(cells)):
        X = X.T.tocsr()
    if X.shape != (len(cells), len(genes)):
        raise ValueError(
            f"Matrix {X.shape}; metadata imply {(len(cells), len(genes))}"
        )
    return {
        "shape": list(map(int, X.shape)),
        "n_cells": int(len(cells)),
        "n_genes": int(len(genes)),
        "counts_read_report": read_report,
    }


def recover_cdata(p, prior_meta):
    X, count_report = read_proseg_counts(p["counts"])
    cell_meta = pd.read_csv(p["cell_metadata"])
    gene_meta = pd.read_csv(p["gene_metadata"])
    if X.shape == (len(gene_meta), len(cell_meta)):
        X = X.T.tocsr()
    if X.shape != (len(cell_meta), len(gene_meta)):
        raise ValueError(
            f"Matrix {X.shape}; expected {(len(cell_meta), len(gene_meta))}"
        )
    if "cell" not in cell_meta or "gene" not in gene_meta:
        raise KeyError("Proseg metadata lacks cell/gene columns.")

    cell_ids = pd.to_numeric(cell_meta["cell"], errors="raise").astype(np.int64)
    expected = pd.Index(np.arange(len(cell_meta), dtype=np.int64))
    if set(cell_ids) != set(expected):
        raise ValueError("Cell IDs are not a complete 0..n_cells-1 sequence.")

    obs = (
        cell_meta.assign(cell=cell_ids)
        .set_index("cell", drop=False)
        .reindex(expected)
    )
    obs.index = obs.index.astype(str)
    obs.index.name = None

    var = gene_meta.copy()
    var.index = var["gene"].astype(str)
    var.index.name = None

    cdata = ad.AnnData(X=X, obs=obs, var=var)
    cdata.var_names_make_unique()
    if "cluster" in cdata.obs and "component" not in cdata.obs:
        cdata.obs["component"] = cdata.obs["cluster"].to_numpy()
    cdata.obsm["spatial"] = cdata.obs[
        ["centroid_x", "centroid_y"]
    ].to_numpy(dtype=np.float32)

    gdf = read_gz_geojson(p["polygons"])
    if "cell" not in gdf:
        raise KeyError("Polygon GeoJSON lacks cell property.")
    gdf["cell"] = pd.to_numeric(gdf["cell"], errors="raise").astype(np.int64)
    gdf = gdf.set_index(gdf["cell"].astype(str), drop=False)
    gdf.index.name = None
    gdf = clear_local_crs(gdf, "Recovered Proseg polygons")

    areas = pd.Series(
        gdf.geometry.area.to_numpy(dtype=float),
        index=gdf.index,
    )
    cdata.obs["cell_area_um2"] = areas.reindex(
        cdata.obs_names
    ).to_numpy(dtype=float)
    cdata.obs["equivalent_2um_bins"] = cdata.obs["cell_area_um2"] / 4.0
    cdata.obs["region"] = "cell_boundaries"

    audit = pd.read_csv(p["prior_audit"], compression="infer")
    audit_lookup = (
        audit.loc[audit["clean_label_id"] > 0]
        .drop_duplicates("clean_label_id")
        .set_index("clean_label_id")
    )
    original = pd.to_numeric(
        cdata.obs["original_cell_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )
    for output_column, audit_column in {
        "prior_qc_keep_bins": "n_qc_keep_bins",
        "prior_qc_drop_bins": "n_qc_drop_bins",
        "prior_qc_sampled_bins": "n_qc_sampled_bins",
        "prior_raw_label_id": "raw_label_id",
    }.items():
        cdata.obs[output_column] = original.map(
            audit_lookup[audit_column]
        ).to_numpy()

    x = np.asarray(prior_meta["x_transform"], dtype=float)
    y = np.asarray(prior_meta["y_transform"], dtype=float)
    linear = np.array([[x[0], x[1]], [y[0], y[1]]], dtype=float)
    offset = np.array([x[2], y[2]], dtype=float)
    inverse = np.linalg.inv(linear)
    mask_xy = (
        np.asarray(cdata.obsm["spatial"]) - offset
    ) @ inverse.T
    cdata.obsm["spatial_stardist_mask_px"] = mask_xy.astype(np.float32)

    add_standard_qc(cdata)
    cdata.uns["comparison_provenance"] = sanitize_none(
        {
            "sample": prior_meta["sample"],
            "source_qc_input": str(p["qc_input"]),
            "prior_mask": str(p["prior_mask"]),
            "prior_audit": str(p["prior_audit"]),
            "native_proseg_zarr": str(p["native_zarr"]),
            "authoritative_recovery": (
                "durable MTX + cell CSV + gene CSV + GeoJSON"
            ),
            "counts_read_report": count_report,
            "pipeline_version": PIPELINE_VERSION,
        }
    )
    return cdata, gdf, count_report


In [6]:
# ---------------------------------------------------------------------
# Atomic output, QC figures, and CRS-safe overlays
# ---------------------------------------------------------------------
def atomic_write_h5ad(adata_obj: ad.AnnData, path: Path):
    temporary = path.with_name(path.stem + ".tmp.h5ad")
    temporary.unlink(missing_ok=True)
    adata_obj.uns = sanitize_none(dict(adata_obj.uns))
    adata_obj.write_h5ad(temporary, compression=H5AD_COMPRESSION)
    check = ad.read_h5ad(temporary)
    if check.shape != adata_obj.shape:
        raise ValueError("H5AD read-back shape mismatch.")
    del check
    temporary.replace(path)


def save_qc_pdf(cdata: ad.AnnData, path: Path, sample: str):
    with PdfPages(path) as pdf:
        fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
        axes[0].hist(np.log10(cdata.obs["total_counts"] + 1), bins=60)
        axes[0].set_title("log10 counts")
        axes[1].hist(
            np.log10(cdata.obs["n_genes_by_counts"] + 1), bins=60
        )
        axes[1].set_title("log10 genes")
        axes[2].hist(cdata.obs["pct_counts_mt"], bins=60)
        axes[2].set_title("% mitochondrial")
        axes[3].hist(cdata.obs["cell_area_um2"].dropna(), bins=60)
        axes[3].set_title("cell area µm²")
        fig.suptitle(sample)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)


def mask_affine_for_shapely(prior_meta):
    x = np.asarray(prior_meta["x_transform"], dtype=float)
    y = np.asarray(prior_meta["y_transform"], dtype=float)
    linear = np.array([[x[0], x[1]], [y[0], y[1]]], dtype=float)
    inverse = np.linalg.inv(linear)
    inverse_offset = -inverse @ np.array([x[2], y[2]], dtype=float)
    return [
        inverse[0, 0],
        inverse[0, 1],
        inverse[1, 0],
        inverse[1, 1],
        inverse_offset[0],
        inverse_offset[1],
    ]


def figure_size(width, height, max_inches):
    if width >= height:
        return max_inches, max(4.0, max_inches * height / width)
    return max(4.0, max_inches * width / height), max_inches


def save_overlays(image_path, gdf, prior_meta, whole_path, preview_path):
    parameters = mask_affine_for_shapely(prior_meta)
    gdf_px = clear_local_crs(gdf, "Overlay input")
    gdf_px.geometry = gdf_px.geometry.map(
        lambda geometry: affine_transform(geometry, parameters)
        if geometry is not None and not geometry.is_empty
        else geometry
    )
    gdf_px = clear_local_crs(gdf_px, "Pixel-space polygons")

    with Image.open(image_path) as image:
        width, height = image.size
        whole = image.convert("RGB")
        whole.thumbnail((WHOLE_IMAGE_MAX_SIDE, WHOLE_IMAGE_MAX_SIDE))
        array = np.asarray(whole)

        fig, ax = plt.subplots(figsize=figure_size(width, height, 12))
        ax.imshow(
            array,
            extent=(0, width, height, 0),
            aspect="equal",
            interpolation="none",
        )
        gdf_px.boundary.plot(
            ax=ax,
            linewidth=0.15,
            color="#00C8FF",
            aspect=None,
        )
        ax.set_xlim(0, width)
        ax.set_ylim(height, 0)
        ax.set_aspect("equal", adjustable="box")
        ax.axis("off")
        fig.savefig(
            whole_path,
            dpi=250,
            bbox_inches="tight",
            pad_inches=0,
        )
        plt.close(fig)

        crop_width = min(PREVIEW_CROP_SIZE_PX, width)
        crop_height = min(PREVIEW_CROP_SIZE_PX, height)
        left = max(0, (width - crop_width) // 2)
        upper = max(0, (height - crop_height) // 2)
        right = left + crop_width
        lower = upper + crop_height
        crop = np.asarray(
            image.crop((left, upper, right, lower)).convert("RGB")
        )

    region = box(left, upper, right, lower)
    subset = gdf_px[
        gdf_px.geometry.notna()
        & gdf_px.geometry.intersects(region)
    ]
    fig, ax = plt.subplots(
        figsize=figure_size(crop_width, crop_height, 10)
    )
    ax.imshow(
        crop,
        extent=(left, right, lower, upper),
        aspect="equal",
        interpolation="none",
    )
    if len(subset):
        subset.boundary.plot(
            ax=ax,
            linewidth=0.45,
            color="#00C8FF",
            aspect=None,
        )
    ax.set_xlim(left, right)
    ax.set_ylim(lower, upper)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")
    fig.savefig(
        preview_path,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close(fig)


def postprocessing_outputs_complete(p: dict[str, Path]) -> bool:
    """
    Return True only when all currently requested post-processing outputs exist.

    This prevents a stale success marker from skipping regeneration when the
    H5AD exists but the boundary Parquet or PNG previews are missing.
    """
    required = [
        p["success"],
        p["cdata"],
        p["boundaries"],
        p["metrics"],
    ]

    if MAKE_QC_PDF:
        required.append(p["qc_pdf"])

    if MAKE_OVERLAYS:
        required.extend(
            [
                p["overlay_whole"],
                p["overlay_preview"],
            ]
        )

    return all(
        path.exists()
        and (
            path.is_dir()
            or path.stat().st_size > 0
        )
        for path in required
    )


def process_sample(row: pd.Series):
    """
    Sequentially recover and post-process one completed Proseg sample.

    In the parallel notebook, the expensive Proseg inference has already been
    completed in Phase A. This function validates and reuses the durable
    MTX/CSV/GeoJSON outputs, then creates the H5AD, boundary Parquet, QC files,
    and image overlays. It does not rerun Proseg when durable outputs validate.
    """
    sample = str(row["sample"])
    p = paths_for_sample(row)

    if (
        postprocessing_outputs_complete(p)
        and not OVERWRITE_POSTPROCESS
    ):
        print("Reusing complete post-processed output:", sample)
        return json.loads(p["success"].read_text())

    for required in [
        p["prior_mask"],
        p["prior_audit"],
        p["prior_metadata"],
        p["image_crop"],
    ]:
        if not required.exists():
            raise FileNotFoundError(required)

    qc_input = build_qc_input(row, p)
    prior_meta = json.loads(
        p["prior_metadata"].read_text()
    )

    command = build_command(
        p,
        qc_input,
        prior_meta,
    )
    command_text = shlex.join(command)
    command_hash = hashlib.sha256(
        command_text.encode()
    ).hexdigest()
    p["command"].write_text(command_text + "\n")

    # Durable Proseg outputs are authoritative. This is the normal route
    # after Phase A of the parallel notebook.
    validation = validate_separate_outputs(p)

    if p["run_status"].exists():
        run_status = json.loads(
            p["run_status"].read_text()
        )
        saved_hash = run_status.get("command_hash")

        if (
            saved_hash not in {None, "", command_hash}
            and not OVERWRITE_PROSEG
        ):
            raise RuntimeError(
                "Durable outputs exist but were produced by a different "
                "Proseg command. Set OVERWRITE_PROSEG=True only when you "
                "intend to replace them."
            )

    print(
        "Durable Proseg outputs validated; "
        "recovering post-processing products without rerunning Proseg."
    )

    cdata, gdf, count_report = recover_cdata(
        p,
        prior_meta,
    )

    atomic_write_h5ad(
        cdata,
        p["cdata"],
    )

    clear_local_crs(
        gdf,
        "Boundary parquet",
    ).to_parquet(
        p["boundaries"]
    )

    metric_columns = [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb",
        "cell_area_um2",
        "equivalent_2um_bins",
        "prior_qc_keep_bins",
        "prior_qc_drop_bins",
        "prior_qc_sampled_bins",
    ]

    cdata.obs[
        [
            column
            for column in metric_columns
            if column in cdata.obs
        ]
    ].to_csv(
        p["metrics"]
    )

    if MAKE_QC_PDF:
        save_qc_pdf(
            cdata,
            p["qc_pdf"],
            sample,
        )

    overlay_status = {
        "requested": bool(MAKE_OVERLAYS),
        "completed": False,
        "error": "",
    }

    if MAKE_OVERLAYS:
        try:
            save_overlays(
                p["image_crop"],
                gdf,
                prior_meta,
                p["overlay_whole"],
                p["overlay_preview"],
            )
            overlay_status["completed"] = True
            p["overlay_error"].unlink(
                missing_ok=True
            )

        except Exception as exc:
            overlay_status["error"] = (
                f"{type(exc).__name__}: {exc}"
            )

            atomic_json(
                {
                    "sample": sample,
                    "stage": "overlay_plotting",
                    "error": overlay_status["error"],
                },
                p["overlay_error"],
            )

            warnings.warn(
                "Cell data, QC outputs, and boundary Parquet were saved, "
                "but overlay generation failed: "
                + overlay_status["error"]
            )

            if OVERLAY_ERRORS_ARE_FATAL:
                raise

    previous_status = {}
    if p["run_status"].exists():
        previous_status = json.loads(
            p["run_status"].read_text()
        )

    success = {
        "sample": sample,
        "completed": True,
        "cdata_h5ad": str(p["cdata"]),
        "cell_boundaries_parquet": str(
            p["boundaries"]
        ),
        "metrics_csv": str(p["metrics"]),
        "qc_pdf": (
            str(p["qc_pdf"])
            if MAKE_QC_PDF
            else ""
        ),
        "boundaries_whole_png": (
            str(p["overlay_whole"])
            if overlay_status["completed"]
            else ""
        ),
        "boundaries_preview_png": (
            str(p["overlay_preview"])
            if overlay_status["completed"]
            else ""
        ),
        "shape": list(map(int, cdata.shape)),
        "total_counts": float(
            cdata.obs["total_counts"].sum()
        ),
        "counts_read_report": count_report,
        "proseg_return_code": previous_status.get(
            "return_code"
        ),
        "command_hash": command_hash,
        "overlay_status": overlay_status,
        "pipeline_version": (
            PIPELINE_VERSION
            + "-parallel-postprocess-fix-v2"
        ),
    }

    atomic_json(
        success,
        p["success"],
    )
    p["failure"].unlink(missing_ok=True)

    print("Wrote:", p["cdata"])
    print("Wrote:", p["boundaries"])
    print("Wrote:", p["metrics"])

    if MAKE_QC_PDF:
        print("Wrote:", p["qc_pdf"])

    if overlay_status["completed"]:
        print("Wrote:", p["overlay_whole"])
        print("Wrote:", p["overlay_preview"])

    del cdata, gdf
    gc.collect()

    return success


In [7]:
# ---------------------------------------------------------------------
# Recovery-only validation and sequential post-processing
# ---------------------------------------------------------------------
def recover_existing_sample(row: pd.Series) -> dict:
    """
    Build final Python outputs from existing durable Proseg files only.

    This function never invokes Proseg and does not depend on a native
    SpatialData Zarr. It is therefore safe after the parallel inference phase
    has completed but the notebook-level post-processing function was missing.
    """
    sample = str(row["sample"])
    p = paths_for_sample(row)

    if (
        postprocessing_outputs_complete(p)
        and not OVERWRITE_POSTPROCESS
    ):
        print("Reusing complete post-processed output:", sample)
        return json.loads(p["success"].read_text())

    for required in [
        p["prior_mask"],
        p["prior_audit"],
        p["prior_metadata"],
        p["image_crop"],
    ]:
        if not required.exists():
            raise FileNotFoundError(
                f"Required StarDist/QC-prior file is absent: {required}"
            )

    # This is the decisive safeguard: no downstream file is created unless
    # the four durable Proseg outputs agree in cell/gene dimensions.
    validation = validate_separate_outputs(p)
    print("Validated durable Proseg outputs:")
    print(json.dumps(validation, indent=2))

    prior_meta = json.loads(
        p["prior_metadata"].read_text()
    )

    cdata, gdf, count_report = recover_cdata(
        p,
        prior_meta,
    )

    # Write and read back the cell-level AnnData atomically.
    atomic_write_h5ad(
        cdata,
        p["cdata"],
    )

    # Proseg coordinates are local microns, not longitude/latitude.
    gdf = clear_local_crs(
        gdf,
        "Boundary parquet",
    )
    gdf.to_parquet(
        p["boundaries"]
    )

    metric_columns = [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb",
        "cell_area_um2",
        "equivalent_2um_bins",
        "prior_qc_keep_bins",
        "prior_qc_drop_bins",
        "prior_qc_sampled_bins",
    ]
    cdata.obs[
        [
            column
            for column in metric_columns
            if column in cdata.obs
        ]
    ].to_csv(
        p["metrics"]
    )

    if MAKE_QC_PDF:
        save_qc_pdf(
            cdata,
            p["qc_pdf"],
            sample,
        )

    overlay_status = {
        "requested": bool(MAKE_OVERLAYS),
        "completed": False,
        "error": "",
    }

    if MAKE_OVERLAYS:
        try:
            save_overlays(
                p["image_crop"],
                gdf,
                prior_meta,
                p["overlay_whole"],
                p["overlay_preview"],
            )
            overlay_status["completed"] = True
            p["overlay_error"].unlink(
                missing_ok=True
            )

        except Exception as exc:
            overlay_status["error"] = (
                f"{type(exc).__name__}: {exc}"
            )
            atomic_json(
                {
                    "sample": sample,
                    "stage": "recovery_overlay_plotting",
                    "error": overlay_status["error"],
                },
                p["overlay_error"],
            )
            warnings.warn(
                "The H5AD, boundary Parquet, metrics, and QC PDF were "
                "written, but overlay generation failed: "
                + overlay_status["error"]
            )
            if OVERLAY_ERRORS_ARE_FATAL:
                raise

    prior_run_status = {}
    if p["run_status"].exists():
        prior_run_status = json.loads(
            p["run_status"].read_text()
        )

    success = {
        "sample": sample,
        "completed": True,
        "recovered_without_proseg_rerun": True,
        "cdata_h5ad": str(p["cdata"]),
        "cell_boundaries_parquet": str(
            p["boundaries"]
        ),
        "metrics_csv": str(p["metrics"]),
        "qc_pdf": (
            str(p["qc_pdf"])
            if MAKE_QC_PDF
            else ""
        ),
        "boundaries_whole_png": (
            str(p["overlay_whole"])
            if overlay_status["completed"]
            else ""
        ),
        "boundaries_preview_png": (
            str(p["overlay_preview"])
            if overlay_status["completed"]
            else ""
        ),
        "shape": list(map(int, cdata.shape)),
        "total_counts": float(
            cdata.obs["total_counts"].sum()
        ),
        "durable_validation": validation,
        "counts_read_report": count_report,
        "proseg_return_code": prior_run_status.get(
            "return_code"
        ),
        "original_command_hash": prior_run_status.get(
            "command_hash",
            "",
        ),
        "overlay_status": overlay_status,
        "pipeline_version": (
            PIPELINE_VERSION
            + "-postprocess-only-recovery-v1"
        ),
    }

    atomic_json(
        success,
        p["success"],
    )
    p["failure"].unlink(missing_ok=True)

    print("Wrote:", p["cdata"])
    print("Wrote:", p["boundaries"])
    print("Wrote:", p["metrics"])
    if MAKE_QC_PDF:
        print("Wrote:", p["qc_pdf"])
    if overlay_status["completed"]:
        print("Wrote:", p["overlay_whole"])
        print("Wrote:", p["overlay_preview"])

    del cdata, gdf
    gc.collect()

    return success


# --------------------------------------------------------------
# Inventory durable outputs before starting recovery.
# --------------------------------------------------------------
inventory_rows = []

for _, row in manifest.iterrows():
    sample = str(row["sample"])
    p = paths_for_sample(row)

    durable_paths = {
        "counts": p["counts"],
        "cell_metadata": p["cell_metadata"],
        "gene_metadata": p["gene_metadata"],
        "polygons": p["polygons"],
    }

    inventory_rows.append(
        {
            "sample": sample,
            **{
                f"{name}_exists": bool(
                    path.exists()
                    and path.stat().st_size > 0
                )
                for name, path in durable_paths.items()
            },
            "h5ad_exists": p["cdata"].exists(),
            "boundaries_parquet_exists": (
                p["boundaries"].exists()
            ),
            "success_marker_exists": p["success"].exists(),
        }
    )

inventory = pd.DataFrame(inventory_rows)
display(inventory)

durable_columns = [
    "counts_exists",
    "cell_metadata_exists",
    "gene_metadata_exists",
    "polygons_exists",
]
ready = inventory[durable_columns].all(axis=1)

print(
    f"Samples with all four durable outputs: "
    f"{int(ready.sum())}/{len(inventory)}"
)

# --------------------------------------------------------------
# Recover samples sequentially.
# --------------------------------------------------------------
results = {}
failures = {}

for _, row in manifest.iterrows():
    sample = str(row["sample"])

    print("\n" + "=" * 90)
    print("Post-processing existing Proseg output:", sample)

    try:
        results[sample] = recover_existing_sample(row)

    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        failures[sample] = error

        p = paths_for_sample(row)
        atomic_json(
            {
                "sample": sample,
                "stage": "postprocess_only_recovery",
                "error": error,
                "traceback": traceback.format_exc(),
            },
            p["failure"],
        )

        traceback.print_exc(limit=30)

        if not CONTINUE_ON_ERROR:
            raise

    finally:
        gc.collect()

# --------------------------------------------------------------
# Refresh all-sample summary and failure files.
# --------------------------------------------------------------
summary = pd.DataFrame(results.values())

summary_path = (
    PROSEG_DERIVED_ROOT
    / "all_samples_proseg_summary.csv"
)
failure_path = (
    PROSEG_DERIVED_ROOT
    / "all_samples_proseg_failures.json"
)
failure_backup = (
    PROSEG_DERIVED_ROOT
    / "all_samples_proseg_failures.before_postprocess_recovery.json"
)

if failure_path.exists() and not failure_backup.exists():
    shutil.copy2(
        failure_path,
        failure_backup,
    )

summary.to_csv(
    summary_path,
    index=False,
)
failure_path.write_text(
    json.dumps(failures, indent=2)
)

print("\nCompleted:", sorted(results))
print("Failures:", json.dumps(failures, indent=2))
print("Updated:", summary_path)
print("Updated:", failure_path)
print("Original failure report backup:", failure_backup)

display(summary)

if failures:
    raise RuntimeError(
        "At least one sample could not be recovered. "
        "The successfully recovered samples remain valid."
    )


,sample,counts_exists,cell_metadata_exists,gene_metadata_exists,polygons_exists,h5ad_exists,boundaries_parquet_exists,success_marker_exists
0,Ada-1,True,True,True,True,False,False,False
1,Ada-3R,True,True,True,True,False,False,False
2,Ada-4R,True,True,True,True,False,False,False
3,Ada-6,True,True,True,True,False,False,False
4,Ada-7,True,True,True,True,False,False,False
5,Ada-8,True,True,True,True,False,False,False
6,Ada-9,True,True,True,True,False,False,False
7,Ada-11R,True,True,True,True,False,False,False
8,Ada-12,True,True,True,True,False,False,False
9,Ada-14R,True,True,True,True,False,False,False


Samples with all four durable outputs: 12/12

Post-processing existing Proseg output: Ada-1
Validated durable Proseg outputs:
{
  "shape": [
    430475,
    19305
  ],
  "n_cells": 430475,
  "n_genes": 19305,
  "counts_read_report": {
    "method": "normal_gzip"
  }
}
Read Ada-1_proseg_qcclass_cell_polygons.geojson.gz: discarding CRS EPSG:4326; coordinates are local microns.
Wrote: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/02_proseg/Ada-1/Ada-1_proseg_qcclass_cdata.h5ad
Wrote: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/02_proseg/Ada-1/Ada-1_proseg_qcclass_cell_boundaries.parquet
Wrote: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/02_proseg/Ada-1/Ada-1_proseg_qcclass_metrics_cells.csv
Wrote: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_r

,sample,completed,recovered_without_proseg_rerun,cdata_h5ad,cell_boundaries_parquet,metrics_csv,qc_pdf,boundaries_whole_png,boundaries_preview_png,shape,total_counts,durable_validation,counts_read_report,proseg_return_code,original_command_hash,overlay_status,pipeline_version
0,Ada-1,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,"[430475, 19305]",272200811.0,"{'shape': [430475, 19305], 'n_cells': 430475, ...",{'method': 'normal_gzip'},0,268fdfc8e7a23505f7217995681bb6082647ff44ca5fd2...,"{'requested': True, 'completed': True, 'error'...",reusable-proseg-durable-output-v1-postprocess-...
1,Ada-3R,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,"[497270, 19308]",281638601.0,"{'shape': [497270, 19308], 'n_cells': 497270, ...",{'method': 'normal_gzip'},0,2b903f63d0be56258f875f65f43c9696edea37e9ee886e...,"{'requested': True, 'completed': True, 'error'...",reusable-proseg-durable-output-v1-postprocess-...
2,Ada-4R,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,"[618975, 19300]",314572682.0,"{'shape': [618975, 19300], 'n_cells': 618975, ...",{'method': 'normal_gzip'},0,d2290450a901e74823c8095d12be2a22c9be89b781dce8...,"{'requested': True, 'completed': True, 'error'...",reusable-proseg-durable-output-v1-postprocess-...
3,Ada-6,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,"[685472, 19308]",271718233.0,"{'shape': [685472, 19308], 'n_cells': 685472, ...",{'method': 'normal_gzip'},0,81560f3c6a1fb39287b7656f892f260299cbe539951a8b...,"{'requested': True, 'completed': True, 'error'...",reusable-proseg-durable-output-v1-postprocess-...
4,Ada-7,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,"[595130, 19310]",311153449.0,"{'shape': [595130, 19310], 'n_cells': 595130, ...",{'method': 'normal_gzip'},0,35a8f067e8a48e4923af4b63ea930f66e8786f8f54c11c...,"{'requested': True, 'completed': True, 'error'...",reusable-proseg-durable-output-v1-postprocess-...
5,Ada-8,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,"[641663, 19316]",321052661.0,"{'shape': [641663, 19316], 'n_cells': 641663, ...",{'method': 'normal_gzip'},0,b28b0a96ce47378804d0f178a59c635ffd5d004958ffce...,"{'requested': True, 'completed': True, 'error'...",reusable-proseg-durable-output-v1-postprocess-...
6,Ada-9,True,True,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBIO-8110_VisiumHD-Adagras...,/stash/data/nonclin/TBI